# Multivariate BOCD: Regime Shifts Across Asset Classes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/multivariate_bocd.ipynb)

**Part 3 of the changepoint detection series:**
- [Part 1: PELT — offline batch changepoint detection](https://sesen.ai/blog/changepoint-detection-regime-shifts)
- [Part 2: Univariate BOCD — online Bayesian changepoint detection](https://sesen.ai/blog/bayesian-online-changepoint-detection)
- **Part 3: Multivariate BOCD — this notebook**

Read the full post: [sesen.ai/blog/multivariate-bocd-regime-shifts-asset-classes](https://sesen.ai/blog/multivariate-bocd-regime-shifts-asset-classes)

In [ ]:
# Run this cell only on Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install yfinance scipy numpy matplotlib pandas --quiet

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm
from scipy.special import gammaln
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "font.size": 11,
})

## 1. Data

We work with five asset classes spanning equities, commodities, and currencies: S&P 500, WTI crude oil, gold, emerging market equities, and the US dollar index. All series are aligned to common trading days (inner join on date index), and we work with daily log returns throughout. Starting in late 2003 gives us a full history through the GFC, COVID, rate-hike cycle, and 2025 geopolitical shocks.

In [ ]:
TICKERS = {
    "^GSPC":    "S&P 500",
    "CL=F":     "Oil (WTI)",
    "GC=F":     "Gold",
    "EEM":      "EM Equities",
    "DX-Y.NYB": "USD Index",
}

raw = yf.download(list(TICKERS.keys()), start="2003-10-01",
                  auto_adjust=True, progress=False)["Close"]
raw.columns = list(TICKERS.values())
log_ret = np.log(raw / raw.shift(1)).dropna()
print(f"{len(log_ret)} aligned trading days")
print(f"From: {log_ret.index[0].date()}  To: {log_ret.index[-1].date()}")
log_ret.head()

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

for ax, col, color in zip(axes, log_ret.columns, colors):
    ax.plot(log_ret.index, log_ret[col].values, linewidth=0.5,
            color=color, alpha=0.85)
    ax.set_ylabel(col, fontsize=9)
    ax.axhline(0, color="black", linewidth=0.4, linestyle="--", alpha=0.4)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(3))

axes[0].set_title("Daily Log Returns — 5 Asset Classes", fontsize=12)
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

## 2. Multivariate BOCD Implementation

The BOCD message-passing loop is identical to the univariate version: at each step we (i) compute the predictive probability of the new observation under every active run-length hypothesis, (ii) propagate mass via growth and changepoint messages, and (iii) renormalise.

What changes is the **underlying predictive model**. In the univariate case we used a Normal-Inverse-Gamma (NIG) conjugate prior, whose posterior predictive is a scalar Student-t. Here we replace it with a **Normal-Inverse-Wishart (NIW)** conjugate prior. The sufficient statistics extend naturally: the scalar mean becomes a vector, the scalar precision becomes a precision matrix, and the posterior predictive becomes a **multivariate Student-t**.

The `MultivariateStudentT` class below tracks one set of NIW parameters per active run-length hypothesis, using vectorised numpy operations so the inner loop stays fast.

In [ ]:
class MultivariateStudentT:
    """NIW conjugate model — predictive: multivariate Student-t."""

    def __init__(self, d, mu0=None, kappa0=1.0, nu0=None, Psi0=None):
        self.d = d
        self.mu0 = mu0 if mu0 is not None else np.zeros(d)
        self.kappa0 = kappa0
        self.nu0 = nu0 if nu0 is not None else float(d + 2)
        self.Psi0 = Psi0 if Psi0 is not None else np.eye(d) * 1e-4
        self.mu    = self.mu0[None, :]
        self.kappa = np.array([kappa0])
        self.nu    = np.array([self.nu0])
        self.Psi   = self.Psi0[None, :, :]

    def log_pred_prob(self, x):
        d  = self.d
        df = self.nu - d + 1
        sf = (self.kappa + 1) / (self.kappa * df)
        S  = self.Psi * sf[:, None, None]
        S  += np.eye(d) * 1e-10
        L  = np.linalg.cholesky(S)
        log_det = 2 * np.sum(np.log(np.diagonal(L, axis1=1, axis2=2)), axis=1)
        diff = x - self.mu
        z    = np.linalg.solve(L, diff[:, :, None]).squeeze(-1)
        quad = np.sum(z**2, axis=1)
        return (
            gammaln((df + d) / 2) - gammaln(df / 2)
            - (d / 2) * np.log(df * np.pi)
            - 0.5 * log_det
            - ((df + d) / 2) * np.log(1 + quad / df)
        )

    def update(self, x):
        diff      = x - self.mu
        new_kappa = self.kappa + 1
        new_nu    = self.nu + 1
        new_mu    = (self.kappa[:, None] * self.mu + x) / new_kappa[:, None]
        outer     = np.einsum("ni,nj->nij", diff, diff)
        new_Psi   = self.Psi + (self.kappa / new_kappa)[:, None, None] * outer
        self.mu    = np.concatenate([self.mu0[None, :],      new_mu  ], axis=0)
        self.kappa = np.concatenate([[self.kappa0],          new_kappa])
        self.nu    = np.concatenate([[self.nu0],             new_nu  ])
        self.Psi   = np.concatenate([self.Psi0[None, :, :], new_Psi ], axis=0)

In [ ]:
def bocd_multivariate(data, hazard_rate=250, kappa0=1.0, nu0=None, Psi0=None):
    """Multivariate BOCD (Adams & MacKay 2007). data: shape (T, d)."""
    T, d = data.shape
    nu0  = nu0 or float(d + 2)
    Psi0 = Psi0 if Psi0 is not None else np.eye(d) * 1e-4
    R       = np.zeros((T + 1, T + 1))
    R[0, 0] = 1.0
    model   = MultivariateStudentT(d, kappa0=kappa0, nu0=nu0, Psi0=Psi0)
    h       = 1.0 / hazard_rate
    maxes   = np.zeros(T)
    p_short = np.zeros(T)
    for t in range(T):
        x    = data[t]
        pred = np.exp(model.log_pred_prob(x))
        R[1:t+2, t+1] = R[:t+1, t] * pred * (1 - h)
        R[0,     t+1] = np.sum(R[:t+1, t] * pred * h)
        ev = R[:t+2, t+1].sum()
        if ev > 0:
            R[:t+2, t+1] /= ev
        model.update(x)
        maxes[t]   = np.argmax(R[:t+2, t+1])
        p_short[t] = R[:min(10, t+2), t+1].sum()
    changepoints = [t for t in range(1, T) if maxes[t] < 5 and maxes[t-1] > 20]
    return R, changepoints, maxes, p_short

## 3. Running the Model

We pass a weakly informative diagonal NIW prior whose scale is calibrated to the historical return variance of each asset. The hazard rate of 250 corresponds to an expected regime duration of approximately one trading year.

In [ ]:
signal = log_ret.values  # (T, 5)
d = signal.shape[1]

# Weakly informative diagonal prior: E[Sigma] ≈ diag(historical variance)
nu0  = float(d + 2)
Psi0 = np.diag(log_ret.std().values**2 * (nu0 - d - 1))

print("Running multivariate BOCD (this may take a few minutes)...")
R, changepoints, maxes, p_short = bocd_multivariate(
    signal, hazard_rate=250, nu0=nu0, Psi0=Psi0
)
print(f"Done. Found {len(changepoints)} changepoints.")

## 4. Results

In [ ]:
dates = log_ret.index
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

fig, axes = plt.subplots(5, 1, figsize=(15, 11), sharex=True)

for ax, col, color in zip(axes, log_ret.columns, colors):
    ax.plot(dates, log_ret[col].values, linewidth=0.5, color=color, alpha=0.8)
    for cp in changepoints:
        ax.axvline(dates[cp], color="red", alpha=0.45, linewidth=0.8)
    ax.set_ylabel(col, fontsize=9)
    ax.axhline(0, color="black", linewidth=0.3, linestyle="--", alpha=0.4)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(3))

axes[0].set_title(
    f"Multivariate BOCD Changepoints (red) — {len(changepoints)} detected",
    fontsize=12,
)
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

In [ ]:
max_rl = 500
T = len(signal)

fig, axes = plt.subplots(2, 1, figsize=(15, 8), sharex=True,
                         gridspec_kw={"height_ratios": [1, 2]})

# Top panel: S&P 500 returns
ax0 = axes[0]
ax0.plot(dates, log_ret["S&P 500"].values, color="black", linewidth=0.5)
for cp in changepoints:
    ax0.axvline(dates[cp], color="red", alpha=0.45, linewidth=0.8)
ax0.set_ylabel("S&P 500 Log Return")
ax0.set_title("Multivariate BOCD — Run-Length Posterior Heatmap")

# Bottom panel: run-length posterior heatmap
ax1 = axes[1]
step  = max(1, T // 2000)
t_idx = np.arange(0, T, step)
R_sub = R[:max_rl, 1:T+1][:, t_idx]
R_sub = np.clip(R_sub, 1e-12, None)

date_nums = mdates.date2num(dates)
extent = [date_nums[t_idx[0]], date_nums[t_idx[-1]], 0, max_rl]
im = ax1.imshow(
    R_sub, aspect="auto", origin="lower", cmap="magma",
    norm=LogNorm(vmin=1e-6, vmax=R_sub.max()), extent=extent,
    interpolation="nearest",
)
ax1.xaxis_date()
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator(3))
ax1.set_ylabel("Run Length")
ax1.set_xlabel("Date")
fig.colorbar(im, ax=ax1, label="P(run length | data)", shrink=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Univariate StudentT and bocd for comparison ─────────────────────────────

class StudentT:
    """Posterior predictive for Normal-Inverse-Gamma conjugate model."""

    def __init__(self, mu0=0, kappa0=1, alpha0=0.01, beta0=0.01):
        self.mu0 = mu0
        self.kappa0 = kappa0
        self.alpha0 = alpha0
        self.beta0 = beta0
        self.mu    = np.array([mu0])
        self.kappa = np.array([kappa0])
        self.alpha = np.array([alpha0])
        self.beta  = np.array([beta0])

    def log_pred_prob(self, x):
        df    = 2 * self.alpha
        scale = self.beta * (self.kappa + 1) / (self.alpha * self.kappa)
        return (
            gammaln((df + 1) / 2) - gammaln(df / 2)
            - 0.5 * np.log(np.pi * df * scale)
            - ((df + 1) / 2) * np.log(1 + (x - self.mu)**2 / (df * scale))
        )

    def update(self, x):
        new_mu    = (self.kappa * self.mu + x) / (self.kappa + 1)
        new_kappa = self.kappa + 1
        new_alpha = self.alpha + 0.5
        new_beta  = (self.beta
                     + self.kappa * (x - self.mu)**2 / (2 * (self.kappa + 1)))
        self.mu    = np.concatenate([[self.mu0],    new_mu   ])
        self.kappa = np.concatenate([[self.kappa0], new_kappa])
        self.alpha = np.concatenate([[self.alpha0], new_alpha])
        self.beta  = np.concatenate([[self.beta0],  new_beta ])


def bocd_univariate(data, hazard_rate=250):
    """Univariate BOCD (Adams & MacKay 2007)."""
    T = len(data)
    R = np.zeros((T + 1, T + 1))
    R[0, 0] = 1.0
    model   = StudentT()
    h       = 1.0 / hazard_rate
    maxes   = np.zeros(T)
    p_short = np.zeros(T)
    for t in range(T):
        x    = data[t]
        pred = np.exp(model.log_pred_prob(x))
        R[1:t+2, t+1] = R[:t+1, t] * pred * (1 - h)
        R[0,     t+1] = np.sum(R[:t+1, t] * pred * h)
        ev = R[:t+2, t+1].sum()
        if ev > 0:
            R[:t+2, t+1] /= ev
        model.update(x)
        maxes[t]   = np.argmax(R[:t+2, t+1])
        p_short[t] = R[:min(10, t+2), t+1].sum()
    return R, maxes, p_short


# Run univariate baseline on S&P 500 only
sp_signal = log_ret["S&P 500"].values
print("Running univariate BOCD on S&P 500...")
R_uni, maxes_uni, p_short_uni = bocd_univariate(sp_signal)
print("Done.")

# Retrieve price series for plotting
raw_prices = yf.download(list(TICKERS.keys()), start="2003-10-01",
                         auto_adjust=True, progress=False)["Close"]
raw_prices.columns = list(TICKERS.values())
prices = raw_prices.loc[log_ret.index]

threshold_low  = 0.05
threshold_mid  = 0.25
threshold_high = 0.50

fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Panel 1: S&P 500 price
ax0 = axes[0]
ax0.plot(dates, prices["S&P 500"].values, color="#1f77b4", linewidth=0.7)
for cp in changepoints:
    ax0.axvline(dates[cp], color="red", alpha=0.4, linewidth=0.8)
ax0.set_ylabel("S&P 500 Price")
ax0.set_title("P(regime < 10 days): Multivariate vs Univariate BOCD")

# Panel 2: Oil price
ax1 = axes[1]
ax1.plot(dates, prices["Oil (WTI)"].values, color="#ff7f0e", linewidth=0.7)
for cp in changepoints:
    ax1.axvline(dates[cp], color="red", alpha=0.4, linewidth=0.8)
ax1.set_ylabel("Oil (WTI) Price")

# Panel 3: P(r < 10) comparison
ax2 = axes[2]
ax2.plot(dates, p_short,     color="red",   linewidth=0.8, label="Multivariate (5 assets)")
ax2.plot(dates, p_short_uni, color="steelblue", linewidth=0.8,
         alpha=0.75, label="Univariate (S&P 500)")
ax2.axhline(threshold_low,  color="black", linestyle=":",  linewidth=0.8, alpha=0.6)
ax2.axhline(threshold_mid,  color="black", linestyle="--", linewidth=0.8, alpha=0.6)
ax2.axhline(threshold_high, color="black", linestyle="-",  linewidth=0.8, alpha=0.6)
ax2.set_ylabel("P(run length < 10 days)")
ax2.set_xlabel("Date")
ax2.set_ylim(0, 1)
ax2.legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_major_locator(mdates.YearLocator(3))

plt.tight_layout()
plt.show()

In [ ]:
# ── Iran War zoom: 2025-10-01 to present ─────────────────────────────────────

zoom_start = pd.Timestamp("2025-10-01")
mask = dates >= zoom_start
dates_zoom = dates[mask]

if len(dates_zoom) == 0:
    print("No data after 2025-10-01 in the downloaded series — skipping zoom.")
else:
    # Normalised prices (rebased to 100 at zoom start)
    prices_zoom = prices.loc[dates_zoom]
    norm_prices = prices_zoom / prices_zoom.iloc[0] * 100

    p_short_zoom     = p_short[mask]
    p_short_uni_zoom = p_short_uni[mask]

    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

    ax0 = axes[0]
    for col, color in zip(norm_prices.columns, colors):
        ax0.plot(dates_zoom, norm_prices[col].values,
                 linewidth=1.2, color=color, label=col)
    ax0.set_ylabel("Normalised Price (base=100)")
    ax0.set_title("Iran War Zoom: Oct 2025 — Present")
    ax0.legend(fontsize=8, ncol=3)
    ax0.axhline(100, color="black", linewidth=0.5, linestyle="--", alpha=0.4)

    ax1 = axes[1]
    ax1.plot(dates_zoom, p_short_zoom,     color="red",      linewidth=1.2,
             label="Multivariate (5 assets)")
    ax1.plot(dates_zoom, p_short_uni_zoom, color="steelblue", linewidth=1.2,
             alpha=0.8, label="Univariate (S&P 500)")
    ax1.axhline(0.05, color="black", linestyle=":",  linewidth=0.8, alpha=0.6,
                label="5% threshold")
    ax1.axhline(0.25, color="black", linestyle="--", linewidth=0.8, alpha=0.6,
                label="25% threshold")
    ax1.axhline(0.50, color="black", linestyle="-",  linewidth=0.8, alpha=0.6,
                label="50% threshold")
    ax1.set_ylabel("P(run length < 10 days)")
    ax1.set_xlabel("Date")
    ax1.set_ylim(0, 1)
    ax1.legend(fontsize=8, ncol=2)

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha="right")

    plt.tight_layout()
    plt.show()

    # ── Threshold crossings ────────────────────────────────────────────────
    for label, arr in [("Multivariate", p_short_zoom), ("Univariate", p_short_uni_zoom)]:
        print(f"\n{label} — first crossings in Iran War period:")
        for threshold in [0.05, 0.25, 0.50]:
            crossed = np.where(arr >= threshold)[0]
            if len(crossed):
                print(f"  P(r<10) >= {threshold:.0%} on {dates_zoom[crossed[0]].date()}")
            else:
                print(f"  P(r<10) never crossed {threshold:.0%}")

In [ ]:
# ── Regime covariance heatmaps ────────────────────────────────────────────────
# Identify regime periods from detected changepoints, find the two longest.

# Build regime segments: (start_idx, end_idx)
boundaries = [0] + changepoints + [len(signal)]
regimes = [(boundaries[i], boundaries[i + 1])
           for i in range(len(boundaries) - 1)]
regime_lengths = [(end - start, start, end) for start, end in regimes]
regime_lengths.sort(reverse=True)
top2 = regime_lengths[:2]

asset_names = list(TICKERS.values())
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (length, start, end) in zip(axes, top2):
    chunk = log_ret.iloc[start:end]
    corr  = chunk.corr()
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r", aspect="auto")
    ax.set_xticks(range(len(asset_names)))
    ax.set_yticks(range(len(asset_names)))
    ax.set_xticklabels(asset_names, rotation=35, ha="right", fontsize=9)
    ax.set_yticklabels(asset_names, fontsize=9)
    # Annotate cells
    for i in range(len(asset_names)):
        for j in range(len(asset_names)):
            ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                    fontsize=8, color="black")
    start_date = dates[start].strftime("%Y-%m-%d")
    end_date   = dates[end - 1].strftime("%Y-%m-%d")
    ax.set_title(f"Regime {start_date} to {end_date}\n({length} days)",
                 fontsize=10)
    fig.colorbar(im, ax=ax, shrink=0.75)

fig.suptitle("Sample Correlation Matrices — Two Longest Detected Regimes",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 5. Exercises

1. **Add a sixth asset**: Replace or add `^VIX` (VIX daily changes). How does adding volatility as a dimension change the detected changepoints?
2. **Hazard rate sensitivity**: Run the multivariate model with `hazard_rate` in [100, 250, 500, 1000]. Plot the number of changepoints vs hazard rate.
3. **Pruning**: Implement run-length pruning — at each step, set `R[r, t+1] = 0` for hypotheses where `R[r, t+1] < 1e-10`, then renormalise. Measure the speedup.
4. **Posterior covariance**: For the current run-length hypothesis (the MAP), extract `Psi[MAP_rl]` and `nu[MAP_rl]` from the `MultivariateStudentT` object and compute the posterior mean covariance `Psi / (nu - d - 1)`. Plot this as a heatmap to see the algorithm's real-time estimate of the current regime's covariance structure.
5. **Non-constant hazard**: Implement a hazard function that increases with run length (older regimes are more likely to end). Does this produce more or fewer changepoints than the constant hazard?